# 03 — ZIT π + 회귀 (경로 B, 정석 Two-Stage)

**정석 Two-Stage 공식** (CLAUDE.md 7-B 준수):
- Stage 1: 01(ZITboost)이 뽑은 `(1 - π_zit)` = P(Y>0 | x)  (분류기 자리를 ZIT 확률로 대체)
- Stage 2: 회귀 모델을 **Y>0 die 서브셋으로만 학습** → `E[Y | Y>0, x]` 예측
- 최종: `pred_B = (1 - π_zit) × reg_pred = P(Y>0|x) × E[Y|Y>0,x] = E[Y|x]`

> HPO objective 는 `(1-π) × reg_pred` 곱셈 **후** unit RMSE 로 계산되므로
> Optuna 가 최적화하는 수식 == 최종 제출 수식 (일관성 보장).

**전제 — 01_zit_only.ipynb 를 먼저 실행**해서 `{zit_only}/oof_die.csv / val_die.csv / test_die.csv` 가 이미 있어야 함.

- `MODEL_NAME` 스위치 (5종 중 하나)
- target_transform: **`'none'` 고정** (곱셈 구조)
- 출력: `4_output/final/zit_plus_reg/{MODEL_NAME}/`

**로컬 + Colab 양쪽 지원.** Colab 사용 시 `GDRIVE_FINAL_ID` + `GDRIVE_ZIT_OUT_ID` 둘 다 필요.

## 1. 환경 설정 + 모듈 import

### Colab 사용 가이드
1. `3_modeling/final/` 를 zip → Drive 업로드 → ID를 `GDRIVE_FINAL_ID`에 입력
2. **01_zit_only 실행 결과 폴더**(`4_output/final/zit_only/`)를 zip → Drive 업로드 → ID를 `GDRIVE_ZIT_OUT_ID`에 입력
   - zip 파일 이름은 자유, 내부에 `oof_die.csv / val_die.csv / test_die.csv` 가 있으면 됨

In [1]:
import os, sys

# ── Colab 사용 시에만 채울 것 ──
GDRIVE_FINAL_ID   = '1HR7LlQmp4n9wGh2WneyVex2mCZ-poiY9'   # ★ final.zip 공유 ID
GDRIVE_ZIT_OUT_ID = ''   # ★ zit_only 결과 폴더 zip 공유 ID

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    # code / dataset / preprocessing
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    # final/ 모듈
    if not os.path.exists('/content/project/3_modeling/final/modules/hpo.py'):
        assert GDRIVE_FINAL_ID, 'GDRIVE_FINAL_ID가 비어있음'
        os.makedirs('/content/project/3_modeling/final', exist_ok=True)
        os.system(f'gdown {GDRIVE_FINAL_ID} -O /content/final.zip')
        os.system('unzip -qo /content/final.zip -d /content/project/3_modeling/final')
    # zit_only 결과 (01 산출물)
    _zit_out = '/content/project/4_output/final/zit_only'
    if not os.path.exists(os.path.join(_zit_out, 'oof_die.csv')):
        assert GDRIVE_ZIT_OUT_ID, 'GDRIVE_ZIT_OUT_ID가 비어있음 — 01 결과 zip 공유 ID 필요'
        os.makedirs(_zit_out, exist_ok=True)
        os.system(f'gdown {GDRIVE_ZIT_OUT_ID} -O /content/zit_only_results.zip')
        os.system(f'unzip -qo /content/zit_only_results.zip -d {_zit_out}')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs
from utils.evaluate import rmse

MODEL_ROOT = os.path.join(PROJECT_ROOT, "3_modeling")
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)
from final.modules import preprocess, hpo, models

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트


## 2. 실험 설정

In [2]:
# ── 모델 선택 (하나만) ──
MODEL_NAME = 'lgbm'        # ★ 'lgbm' | 'xgb' | 'catboost' | 'et' | 'enet'
assert MODEL_NAME in {'lgbm', 'xgb', 'catboost', 'et', 'enet'}

# ── 실험 식별 ──
# 003: search space (zitreg variant) + TARGET_TRANSFORM(log1p) + position 5cols 추가 → 신규 study
EXP_ID   = f'zitreg-{MODEL_NAME}-005'
EXP_MEMO = f'경로 B — ZIT.(1-π) + {MODEL_NAME} × (1-π) + log1p + position cols + zitreg space'
USER     = 'jh'

# ── Optuna 예산 ──
N_TRIALS = 10
N_FOLDS  = 5

# ── REUSE 모드 (기존 best_params.json 재사용 → HPO 스킵, refit만 실행) ──
REUSE_BEST_PARAMS     = False   # ★ True: 기존 HP 재사용 / False: 정상 HPO
PREV_BEST_PARAMS_PATH = ''      # 로컬: 비우면 OUT_DIR/best_params.json 자동 사용 (cell 10 fallback). 다른 위치 쓸 때만 지정.
GDRIVE_PREV_PARAMS_ID = ''      # Colab: 해당 JSON을 Drive 업로드 후 공유 ID

# ── RESUME 모드 (기존 Optuna study에 trial 이어붙임) ──
# REUSE_BEST_PARAMS=False 일 때만 적용. True면 같은 study_name/storage(DB_PATH)에 N_TRIALS개 추가 누적.
# False면 신규 study 생성 (중복 시 DuplicatedStudyError로 안전 차단).
RESUME_STUDY = False   # ★ True: 같은 study 이어달리기 / False: 신규 study

# ── 타깃 처리 ──
# log1p: y → log1p(y) 학습, 예측 시 expm1 + clip(0+) 역변환 (baseline two_stage 와 동일).
#   → (1-π_zit) 곱셈은 expm1 역변환 후 적용 (hpo._eval_split_rmse / refit_best 둘 다 같은 순서).
TARGET_TRANSFORM = 'log1p'   # ★ 'none' | 'log1p'
CLIP_Y_EXTREME   = True

# ── 경로 설정 ──
PATH_A_DIR = os.path.join(OUTPUT_DIR, 'final', 'zit_only')        # 01 결과 위치
OUT_DIR    = os.path.join(OUTPUT_DIR, 'final', 'zit_plus_reg', MODEL_NAME)
DB_PATH    = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# 01 결과 존재 확인
for f in ['oof_die.csv', 'val_die.csv', 'test_die.csv']:
    _p = os.path.join(PATH_A_DIR, f)
    assert os.path.exists(_p), f'{_p} 없음 — 01_zit_only.ipynb 먼저 실행'

# ── 전처리 파라미터 override ──
# 1차 Two-Stage log1p-trial top-50 mode 기반 — log1p 전환에 맞춰 PARAMS도 log1p 모드로 갱신해야 할 수 있음.
# (현재는 none top-50 mode 그대로 유지 — 추후 1차 결과 재집계 시 변경)
PARAMS = {
    'missing_threshold':          0.4,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.98,
    'post_impute_corr_keep_by':   'std',
}

# ── 디바이스 ──
# models.DEVICE = 'gpu'

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'MODEL_NAME (Stage 2 reg): {MODEL_NAME}')
print(f'Stage 1 (π 소스): {PATH_A_DIR}')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'REUSE_BEST_PARAMS={REUSE_BEST_PARAMS} | RESUME_STUDY={RESUME_STUDY}')
print(f'OUT_DIR={OUT_DIR}')

EXP: zitreg-lgbm-005 | USER: jh
MODEL_NAME (Stage 2 reg): lgbm
Stage 1 (π 소스): c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\zit_only
N_TRIALS=10, N_FOLDS=5
TARGET_TRANSFORM=log1p | CLIP_Y_EXTREME=True
REUSE_BEST_PARAMS=False | RESUME_STUDY=False
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\zit_plus_reg\lgbm


## 3. 데이터 로드 + target clip + 전처리

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']
print(f'[전처리 완료] feat_cols: {len(feat_cols_clean)}')

# ── Position 메타 피처 추가 (baseline reg_level='position'와 동일) ──
# POSITION_COL ordinal(1~4) + p_1..p_4 OHE — 트리는 ordinal split, 선형은 OHE coef.
from utils.config import POSITION_COL
for _df in (xs_train, xs_val, xs_test):
    for _p in (1, 2, 3, 4):
        _df[f'p_{_p}'] = (_df[POSITION_COL] == _p).astype('int8')
_pos_extra_cols = [POSITION_COL] + [f'p_{_p}' for _p in (1, 2, 3, 4)]
feat_cols_clean = feat_cols_clean + _pos_extra_cols
print(f'[position 5 cols 추가] feat_cols: {len(feat_cols_clean) - 5} → {len(feat_cols_clean)}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
xs: (174572, 1091), feat_cols: 1087
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=40%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,7

## 4. 01(ZIT)의 (1-π_zit) 로드 — 경로 B 핵심

- train: OOF (1-π) — die 순서는 전처리된 xs_train에 맞춰 DIE_KEY_COL로 자동 정렬
- val/test: fold 평균 (1-π)

In [4]:
# ── 01(경로 A)의 best_params.json 로드 → fold 정렬 일관성 검증 ──
import hashlib, json as _json
with open(os.path.join(PATH_A_DIR, 'best_params.json'), encoding='utf-8') as _f:
    _path_a_meta = _json.load(_f)

# 03 현재의 unit_ids_hash 계산 (save_artifacts와 동일 알고리즘)
_uid = ys_input['train'][KEY_COL].unique()
_uid_hash = hashlib.sha1(','.join(map(str, _uid)).encode('utf-8')).hexdigest()

assert _path_a_meta.get('unit_ids_hash') == _uid_hash, (
    f'01↔03 unit_ids mismatch — '
    f'01({(_path_a_meta.get("unit_ids_hash") or "?")[:8]}) vs '
    f'03({_uid_hash[:8]}). SEED/CLIP/전처리 설정이 달라 (1-π) OOF 정렬이 깨집니다.'
)
assert _path_a_meta.get('n_folds') == N_FOLDS, (
    f'01 N_FOLDS={_path_a_meta.get("n_folds")} ≠ 03 N_FOLDS={N_FOLDS}'
)
# CLIP / SEED 일관성 (assert — fold 의미·재현성 깨짐 방지)
_a_clip = (_path_a_meta.get('study_meta') or {}).get('clip_y_extreme')
if _a_clip is not None:
    assert _a_clip == CLIP_Y_EXTREME, (
        f'01 CLIP_Y_EXTREME={_a_clip} ≠ 03 CLIP_Y_EXTREME={CLIP_Y_EXTREME} — '
        f'곱셈 multiplier 의미가 깨집니다.'
    )
_a_seed = (_path_a_meta.get('study_meta') or {}).get('seed')
if _a_seed is not None:
    assert _a_seed == SEED, (
        f'01 SEED={_a_seed} ≠ 03 SEED={SEED} — fold 분할이 바뀌어 OOF 의미가 어긋납니다.'
    )
print(f'[01↔03 fold 일관성] unit_ids_hash 일치, N_FOLDS={N_FOLDS} 일치')

# ── (1-π_zit) 로드 (split 명시 → 길이·키셋 엄격 검증) ──
omp_train = hpo.load_extra_feature_from_path(PATH_A_DIR, xs_train, feature_col='one_minus_pi', split='train')
omp_val   = hpo.load_extra_feature_from_path(PATH_A_DIR, xs_val,   feature_col='one_minus_pi', split='val')
omp_test  = hpo.load_extra_feature_from_path(PATH_A_DIR, xs_test,  feature_col='one_minus_pi', split='test')

print(f'(1-π) shapes: train {omp_train.shape}, val {omp_val.shape}, test {omp_test.shape}')
print(f'(1-π) summary: train mean={omp_train.mean():.4f} std={omp_train.std():.4f} '
      f'min={omp_train.min():.4f} max={omp_train.max():.4f}')

[01↔03 fold 일관성] unit_ids_hash 일치, N_FOLDS=5 일치
[load_extra_feature] oof_die.csv(train) → one_minus_pi (n=104748)
[load_extra_feature] val_die.csv(val) → one_minus_pi (n=34908)
[load_extra_feature] test_die.csv(test) → one_minus_pi (n=34916)
(1-π) shapes: train (104748,), val (34908,), test (34916,)
(1-π) summary: train mean=0.5749 std=0.2360 min=0.0109 max=0.9413


## 5. Optuna HPO — reg 모델 (추가 피처 = (1-π_zit))

In [5]:
import json

best_params_for_refit = None
already_resolved = False
study = None

# ── target transform/inverse 등록 (baseline two_stage 와 동일) ──
# log1p:    y → log1p(y) 학습, 예측 후 expm1 + clip(0+) → multiplier (1-π) 곱셈
# none:     transform 미사용 (기존 003 이전 동작)
if TARGET_TRANSFORM == 'log1p':
    target_transform_fn = lambda y: np.log1p(np.asarray(y))
    target_inverse_fn   = lambda y: np.clip(np.expm1(np.asarray(y)), 0.0, None)
    print('[log1p] transform/inverse 등록 (ys_input 은 원본 유지, expm1 → ×(1-π) 순서)')
elif TARGET_TRANSFORM == 'none':
    target_transform_fn = None
    target_inverse_fn   = None
    print('[none] transform 미사용')
else:
    raise ValueError(f"TARGET_TRANSFORM={TARGET_TRANSFORM!r} 미지원 ('log1p' | 'none')")

study_meta_for_save = {
    'exp_id':            EXP_ID,
    'exp_memo':          EXP_MEMO,
    'user':              USER,
    'model_name':        MODEL_NAME,
    'stage1_source':     PATH_A_DIR,
    'target_transform':  TARGET_TRANSFORM,
    'clip_y_extreme':    CLIP_Y_EXTREME,
    'effective_pp_params': pp['effective_params'],
    'n_trials':          N_TRIALS,
    'n_folds':           N_FOLDS,
    'seed':              SEED,
    'reuse_best_params': REUSE_BEST_PARAMS,
    'resume_study':      RESUME_STUDY,
    'two_stage_mode':    'classic+A4',   # Stage2=Y>0 only, (1-π) 피처 추가, final=(1-π)×reg
    'multiplier':        'one_minus_pi (from path A)',
    'space_variant':     'zitreg',       # models.SEARCH_SPACES_ZITREG (lgbm 확장 / 나머지 4개 default 복사)
}

if REUSE_BEST_PARAMS:
    # Colab: ID 있으면 gdown으로 JSON 다운로드
    try:
        import google.colab
        if GDRIVE_PREV_PARAMS_ID and not PREV_BEST_PARAMS_PATH:
            _dl = '/content/prev_best_params.json'
            os.system(f'gdown {GDRIVE_PREV_PARAMS_ID} -O {_dl}')
            PREV_BEST_PARAMS_PATH = _dl
    except ImportError:
        pass
    # 로컬 (or Colab GDRIVE_PREV_PARAMS_ID 미지정): 비어있으면 OUT_DIR/best_params.json 자동 사용
    if not PREV_BEST_PARAMS_PATH:
        PREV_BEST_PARAMS_PATH = os.path.join(OUT_DIR, 'best_params.json')
    assert PREV_BEST_PARAMS_PATH and os.path.exists(PREV_BEST_PARAMS_PATH), \
        f'PREV_BEST_PARAMS_PATH 파일 없음: {PREV_BEST_PARAMS_PATH}'
    with open(PREV_BEST_PARAMS_PATH, encoding='utf-8') as f:
        _meta = json.load(f)
    best_params_for_refit = _meta['best_params_resolved']
    already_resolved = True
    study_meta_for_save['prev_best_params_path'] = PREV_BEST_PARAMS_PATH
    print(f'[REUSE] HPO 스킵, best_params 로드: {PREV_BEST_PARAMS_PATH}')
    print(f'  model_name(prev) = {_meta.get("model_name")}')
else:
    # 정석 Two-Stage HPO:
    #   y_positive_only=True  → Stage 2 회귀는 Y>0 die 만 학습 (E[Y|Y>0,x])
    #   multiplier_train       → objective RMSE 계산 시 (1-π_zit) 곱셈 적용
    #   space_variant='zitreg' → zitreg-lgbm-002 분석 기반 확장 search space
    #   target_transform_fn   → log1p(y) 학습 후 expm1 → multiplier 순서로 RMSE 평가
    res = hpo.run_hpo(
        xs_train=xs_train,
        ys_train_unit=ys_input['train'],
        feat_cols=feat_cols_clean,
        model_name=MODEL_NAME,
        n_trials=N_TRIALS,
        n_folds=N_FOLDS,
        y_positive_only=True,
        multiplier_train=omp_train,
        extra_feature_train=('one_minus_pi', omp_train),  # plan §4.2 A4: reg 입력에 (1-π_zit) 피처 추가
        space_variant='zitreg',  # ★ lgbm: num_leaves/max_depth/n_estimators 확장 + tweedie-only objective
        target_transform_fn=target_transform_fn,
        target_inverse_fn=target_inverse_fn,
        study_name=EXP_ID,
        storage=f'sqlite:///{DB_PATH}',
        resume_study=RESUME_STUDY,   # ★ True: 같은 study_name DB에 N_TRIALS개 추가 / False: 신규 (중복 시 DuplicatedStudyError)
        user_attrs=study_meta_for_save,
        # ★ trial별 holdout RMSE 기록 — val/test 도 동일한 (1-π)×reg 최종 공식으로 평가
        xs_val=xs_val,   ys_val_unit=ys_input['validation'],
        multiplier_val=omp_val,
        extra_feature_val=('one_minus_pi', omp_val),
        xs_test=xs_test, ys_test_unit=ys_input['test'],
        multiplier_test=omp_test,
        extra_feature_test=('one_minus_pi', omp_test),
    )
    study                 = res['study']
    best_params_for_refit = res['best_params']
    study_meta_for_save['hpo_best_value'] = float(res['best_value'])
    print(f'\n[HPO 완료] best OOF unit RMSE = {res["best_value"]:.6f}')
    print(f'  (이미 (1-π)×reg_pred 곱셈 포함한 최종 unit RMSE)')
    print(f'best_params = {best_params_for_refit}')

[log1p] transform/inverse 등록 (ys_input 은 원본 유지, expm1 → ×(1-π) 순서)


[I 2026-04-26 15:00:59,393] A new study created in RDB with name: zitreg-lgbm-005


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-04-26 15:01:21,791] Trial 0 finished with value: 0.0060541939920667025 and parameters: {'n_estimators': 1998, 'learning_rate': 0.24517932047070642, 'num_leaves': 565, 'max_depth': 13, 'min_child_samples': 66, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.15227525095137953, 'reg_alpha': 1.6175449623854197, 'reg_lambda': 0.002570603566117598, 'min_split_gain': 0.0003139575515829447, 'path_smooth': 1.0292247147901223, 'objective': 'tweedie_1.2'}. Best is trial 0 with value: 0.0060541939920667025.
[I 2026-04-26 15:02:03,597] Trial 1 finished with value: 0.005914440691843845 and parameters: {'n_estimators': 1080, 'learning_rate': 0.01737635693697876, 'num_leaves': 407, 'max_depth': 10, 'min_child_samples': 120, 'subsample': 0.8059264473611898, 'colsample_bytree': 0.22554447458683766, 'reg_alpha': 5.870705003258033e-06, 'reg_lambda': 1.9826980964985924e-05, 'min_split_gain': 2.970570135694147e-07, 'path_smooth': 39.25879806965068, 'objective': 'tweedie_1.5'}. Best is trial 1

## 6. Best trial 재학습 + (1-π) × reg_pred 적용

In [6]:
# 정석 Two-Stage refit:
#   y_positive_only=True       → Stage 2 학습 데이터에서 Y==0 die 제외
#   target_transform_fn/inverse → log1p 학습 + expm1 역변환 (cell 5 와 동일)
#   multiplier_*                → 예측 후 (1-π_zit) 곱셈을 refit_best 내부에서 처리
#   (수동 곱셈/재집계 코드 불필요 — refit_best 가 oof_pred_die/val_pred_die/test_pred_die
#    를 이미 expm1 → ×(1-π) 까지 적용된 최종 예측으로 반환)
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    best_params=best_params_for_refit,
    already_resolved=already_resolved,
    n_folds=N_FOLDS,
    y_positive_only=True,
    multiplier_train=omp_train,
    multiplier_val=omp_val,
    multiplier_test=omp_test,
    extra_feature_train=('one_minus_pi', omp_train),  # plan §4.2 A4: HPO와 동일하게 reg 입력에 (1-π) 피처 추가
    extra_feature_val=('one_minus_pi', omp_val),
    extra_feature_test=('one_minus_pi', omp_test),
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

# 최종 OOF / val / test unit RMSE (refit_best 내부에서 곱셈이 이미 적용된 상태)
y_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
oof_u  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_true.index]
oof_rmse = float(np.sqrt(np.mean((oof_u.values - y_true.values)**2)))

y_val_true  = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
val_u       = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
val_rmse    = float(np.sqrt(np.mean((val_u.values - y_val_true.values)**2)))

y_test_true = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
test_u      = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]
test_rmse   = float(np.sqrt(np.mean((test_u.values - y_test_true.values)**2)))

print(f'\n[Refit 완료 — 정석 Two-Stage]')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}  (= (1-π)×reg 최종값 기준)')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')
if not REUSE_BEST_PARAMS and study is not None:
    print(f'  HPO best = {res["best_value"]:.6f}  (동일 기준으로 Optuna 최적화한 값)')
print(f'  fold_models: {len(final["fold_models"])}개')

[refit fold 1/5] tr_units=20949, vl_units=5238
[refit fold 2/5] tr_units=20949, vl_units=5238
[refit fold 3/5] tr_units=20950, vl_units=5237
[refit fold 4/5] tr_units=20950, vl_units=5237
[refit fold 5/5] tr_units=20950, vl_units=5237

[Refit 완료 — 정석 Two-Stage]
  OOF  unit RMSE = 0.005711  (= (1-π)×reg 최종값 기준)
  val  unit RMSE = 0.005872
  test unit RMSE = 0.008495
  HPO best = 0.005711  (동일 기준으로 Optuna 최적화한 값)
  fold_models: 5개


## 7. 아티팩트 저장

저장되는 예측값은 **곱셈 적용된 최종 Two-Stage 값**이다.

In [7]:
# ── Postprocess 설정 — 곱셈이 끝난 die 예측에 agg + zero_clip 만 적용 ──
# (π threshold는 이미 (1-π) 곱셈으로 반영되었으므로 off)
POSTPROCESS_CONFIG = {
    'agg_methods':      ('mean', 'median', 'max', 'min', 'trimmed_mean', 'weighted'),
    'zero_clip_range':  (0.001, 0.015),
    'zero_clip_step':   0.001,
    'use_pi_threshold': False,
}

# extra_feature_name='one_minus_pi': reg 입력에 (1-π_zit) 1개 컬럼 추가된 상태 (plan §4.2 A4).
# 최종 예측은 refit_best 내부에서 (1-π_zit) × reg_pred 곱셈까지 완료된 값.
hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name='one_minus_pi',
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],   # ★ val_die.csv / val_unit.csv 에 health
    y_test_unit=ys_input['test'],        # ★ test_die.csv / test_unit.csv 에 health
    postprocess_config=POSTPROCESS_CONFIG,
    study_meta=study_meta_for_save,
)

for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:30s}  {size_kb:10,.1f} KB')
# ── Colab → 로컬 자동 다운로드 (로컬은 자동 skip) ──
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', f'{MODEL_NAME}_{EXP_ID}_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
        print(f'[브라우저 다운로드 트리거] {os.path.basename(_zip_path)}')
    except Exception as _e:
        # VS Code/Jupyter 등 일부 환경에서 files.download 실패 시 FileLink로 대체
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭해 수동 다운로드')
        display(FileLink(_zip_path))
except ImportError:
    pass  # 로컬: skip


[Aggregation] RMSEs: {'mean': 0.005711, 'median': 0.005701, 'max': 0.0061, 'min': 0.005569, 'trimmed_mean': 0.005701, 'weighted': 0.005711}
[Aggregation] best=min (0.005569)
[zero_clip] best=0.0010 (0.005572)
[Postprocess] best_agg=min, pi_th=None, zero_clip=0.0010, train_rmse=0.005572
[save_artifacts] c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\zit_plus_reg\lgbm 저장 완료 (fold_models.pkl + best_params.json + 6 CSV, unit=tuned)
  best_params.json                      10.3 KB
  fold_models.pkl                  130,158.5 KB
  oof_die.csv                        5,562.2 KB
  oof_unit.csv                         903.5 KB
  optuna_jh_zitreg-lgbm-001.db         112.0 KB
  optuna_jh_zitreg-lgbm-002.db         200.0 KB
  optuna_jh_zitreg-lgbm-004.db         128.0 KB
  optuna_jh_zitreg-lgbm-005.db         128.0 KB
  test_die.csv                       1,852.8 KB
  test_unit.csv                        303.6 KB
  val_die.csv                        1,852.9 KB
  val_unit.csv                       

## 8. 요약

In [8]:
print(f'★ 경로 B (ZIT π + 회귀)')
print(f'  EXP_ID       : {EXP_ID}')
print(f'  Stage 1 소스 : {PATH_A_DIR}')
print(f'  Stage 2 모델 : {MODEL_NAME}')
print(f'  transform    : {TARGET_TRANSFORM} (고정)')
print(f'  REUSE 모드   : {REUSE_BEST_PARAMS}')
print(f'  OOF RMSE     : {oof_rmse:.6f}  (Two-Stage 최종)')
print(f'  val RMSE     : {val_rmse:.6f}')
print(f'  test RMSE    : {test_rmse:.6f}')
if not REUSE_BEST_PARAMS and study is not None:
    print(f'  reg HPO best : {res["best_value"]:.6f}  (reg 단독)')
    print(f'  trials       : {len(study.trials)}')
print(f'  feat_cols    : {len(feat_cols_clean)}+1 (one_minus_pi 추가)')
print(f'  저장 경로    : {OUT_DIR}')
print(f'\n다른 모델도 돌리려면 MODEL_NAME 바꿔서 재실행.')

★ 경로 B (ZIT π + 회귀)
  EXP_ID       : zitreg-lgbm-005
  Stage 1 소스 : c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\zit_only
  Stage 2 모델 : lgbm
  transform    : log1p (고정)
  REUSE 모드   : False
  OOF RMSE     : 0.005711  (Two-Stage 최종)
  val RMSE     : 0.005872
  test RMSE    : 0.008495
  reg HPO best : 0.005711  (reg 단독)
  trials       : 10
  feat_cols    : 578+1 (one_minus_pi 추가)
  저장 경로    : c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\zit_plus_reg\lgbm

다른 모델도 돌리려면 MODEL_NAME 바꿔서 재실행.
